In [0]:
user_name = "kumar@vcloudmatesolutions.com"

salesorders_filename = "sales_orders.csv"
products_filename = "products.csv"

salesorders_filePath = (
    "dbfs:/FileStore/shared_uploads/" + user_name + "/" + salesorders_filename
)
products_filePath = (
    "dbfs:/FileStore/shared_uploads/" + user_name + "/" + products_filename
)

In [0]:
salesordersDF = spark.read.csv(salesorders_filePath, header=True, inferSchema=True)
productsDF = spark.read.csv(products_filePath, header=True, inferSchema=True)

In [0]:
display(salesordersDF)
display(productsDF)

order_id,order_date,customer_id,product_id,quantity,total_amount
1,2023-04-15,C045,P040,1,80.19
2,2023-03-02,C004,P045,4,144.64
3,2023-03-11,C024,P034,7,461.16
4,2023-09-29,C048,P039,9,501.03
5,2023-04-25,C008,P043,1,75.96


product_id,price
P040,80.19
P045,36.16
P034,65.88
P039,55.67
P043,75.96


In [0]:
# Source - Batch
# spark.read.format('parquet').load('location').write('delta')

# Source - Streaming
# spark.readStream.format('parquet').load('location').writeStream('delta')

In [0]:
display(salesordersDF.isStreaming)
display(productsDF.isStreaming)


False

False

In [0]:
salesordersDF.write.format("parquet").save("/tmp/salesorders", mode="overwrite")
productsDF.write.format("parquet").save("/tmp/products", mode="overwrite")

In [0]:
salesorders_streamingDF = spark.readStream.schema(salesordersDF.schema)\
                                          .parquet("/tmp/salesorders")
                                           
products_streamingDF = spark.readStream.schema(productsDF.schema)\
                                          .parquet("/tmp/products")

In [0]:
display(salesorders_streamingDF.isStreaming)
display(products_streamingDF.isStreaming)

True

True

In [0]:
salesorder_product_joinedDF = salesorders_streamingDF.join(products_streamingDF,'product_id')

In [0]:
display(salesorder_product_joinedDF.isStreaming)

True

In [0]:
tableName = 'salesorders_tbl100'
checkpointLocation = '/tmp/_checkpoint'

In [0]:
query = (
    salesorder_product_joinedDF.writeStream.toTable(
                tableName=tableName,
                outputMode="append", 
                checkpointLocation=checkpointLocation
                )
)

In [0]:
display(dbutils.fs.ls('/user/hive/warehouse/'))

path,name,size,modificationTime
dbfs:/user/hive/warehouse/salesorders_tbl2/,salesorders_tbl2/,0,0


In [0]:
query.stop()

In [0]:
display(spark.sql('SELECT product_id,SUM(quantity) AS ItemsSoldQty FROM salesorders_tbl100 GROUP BY product_id'))

product_id,ItemsSoldQty
P040,9
P043,9
P045,36
P034,63
P039,81
